## Postcode district to BRMA mapping

The UBDC lookup maps 2.94 million unit postcodes to Broad Rental Market Areas.
Properties are identified by postcode district, which may span more than one
BRMA. This notebook derives a district-level mapping by modal assignment and
records the modal share so that ambiguous districts can be excluded.

Output: `data/processed/district_brma.csv`

In [3]:
import pandas as pd

# Only the columns needed; the full file is 127 MB
lookup = pd.read_csv('../data/raw/pcode_brma_lookup_v2.2.csv', usecols=['country', 'PCDS', 'brma_name'], dtype=str)

print(lookup.shape)
print(lookup.head())

(2936738, 3)
  country     PCDS         brma_name
0       E  AL1 1AA  South West Herts
1       E  AL1 1AB  South West Herts
2       E  AL1 1AD  South West Herts
3       E  AL1 1AE  South West Herts
4       E  AL1 1AF  South West Herts


In [4]:
# England only, and the district is the part before the space in a full postcode
eng = lookup[lookup['country'] == 'E'].copy()
eng['districts'] = eng['PCDS'].str.split().str[0]

print("England Postcodes :", len(eng))
print("England Districts :", eng['districts'].nunique())

England Postcodes : 2247615
England Districts : 2322


### Modal assignment

Each district is assigned the BRMA containing the greatest number of its unit
postcodes. The modal share records how much of the district that BRMA covers,
so districts split across areas can be identified and, where the share is low,
excluded from cross-area comparison.

In [6]:
# Count unit postcodes per district/BRMA pair
counts = (eng.groupby(['districts', 'brma_name']).size().reset_index(name = 'postcodes'))

# Total postcodes in each district
totals = counts.groupby('districts')['postcodes'].sum()

# The BRMA with most postcodes in one district
modal = counts.sort_values('postcodes', ascending=False).groupby('districts').first()
modal['total'] = totals
modal['modal_share'] = modal['postcodes'] / modal['total']

print(modal.shape)
print(modal.head())

(2322, 4)
                  brma_name  postcodes  total  modal_share
districts                                                 
AL1        South West Herts       2003   2003     1.000000
AL10       South East Herts       1195   1195     1.000000
AL2        South West Herts        984    984     1.000000
AL3        South West Herts       1305   1306     0.999234
AL4        South West Herts       1048   1058     0.990548


### Saving the district mapping

The derived mapping is written to `data/processed/` so that agents can look up
a BRMA without loading the 127 MB source file. The modal share is retained so
that the Affordability agent can flag or exclude districts where the assignment
is unreliable.

In [7]:
# How clean is the mapping?
print("Districts mapping entirely to one BRMA:", (modal['modal_share'] == 1.0).sum())
print("Modal share below 0.80:", (modal['modal_share'] < 0.80).sum())
print()
print("Modal share distribution:")
print(modal['modal_share'].describe().round(3))

Districts mapping entirely to one BRMA: 1375
Modal share below 0.80: 136

Modal share distribution:
count    2322.000
mean        0.968
std         0.085
min         0.426
25%         0.992
50%         1.000
75%         1.000
max         1.000
Name: modal_share, dtype: float64


In [8]:
out = modal[['brma_name', 'modal_share', 'total']].reset_index()
out.columns = ['district', 'brma', 'modal_share', 'postcodes']

out.to_csv('../data/processed/district_brma.csv', index=False)
print("Saved:", out.shape)
print(out.head())

Saved: (2322, 4)
  district              brma  modal_share  postcodes
0      AL1  South West Herts     1.000000       2003
1     AL10  South East Herts     1.000000       1195
2      AL2  South West Herts     1.000000        984
3      AL3  South West Herts     0.999234       1306
4      AL4  South West Herts     0.990548       1058


In [13]:
# Which of the project's districts have unreliable assignment?
listings_districts = pd.read_csv('../data/processed/listings_clean.csv') \
    if False else None   # not saved yet

# Use the modal table directly for London BRMAs
london = modal[modal['brma_name'].str.contains('London', na = False)]
print("Districts in London BRMAs :", len(london))
print("Beloe 0.80 modal share :", (london['modal_share'] < 0.80).sum())
print()
print(london[london['modal_share'] < 0.80].sort_values('modal_share').head(20))

Districts in London BRMAs : 311
Beloe 0.80 modal share : 19

                         brma_name  postcodes  total  modal_share
districts                                                        
N4              Inner North London        779   1544     0.504534
UB6              North West London        728   1365     0.533333
SW1P                Central London        880   1594     0.552070
SW8        Inner South East London        881   1502     0.586551
TW2        Outer South West London        553    889     0.622047
UB5              North West London        939   1422     0.660338
W10             Inner North London        751   1134     0.662257
TN16       Outer South East London        613    916     0.669214
UB10             Outer West London        642    952     0.674370
SW16       Inner South East London       1656   2252     0.735346
SE12       Outer South East London        611    828     0.737923
TW7              Outer West London        994   1347     0.737936
N1             